In [2]:
import sys

sys.path.append("..")

import os
from datetime import datetime
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from nnspike.constants import RELATIVE_POSITION_SCALE
from nnspike.data import RegressionDataset, balance_dataset
from nnspike.models import NvidiaModelRegression

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_course = "right"
date_label = datetime.today().strftime("%m%d")

print(f"Training Course: {train_course}; Date Label: {date_label}")

Training Course: right; Date Label: 1008


## Loading Dataset

In [3]:
label_paths = glob("../storage/labels/*.csv")
label_paths = [path for path in label_paths if os.path.basename(path)[:8] < "20250801"]

df = pd.DataFrame()
for label_path in label_paths:
    label_df = pd.read_csv(label_path)
    df = pd.concat([df, label_df])

# Drop the unlabeled rows
df = df.dropna(subset=["target_x", "mode"])

df["mode"] = df["mode"].astype(int)

print(f"Total number of training records: {len(df)}")
print(f"Unique behavior mode: {df['mode'].unique()}")

Total number of training records: 47143
Unique behavior mode: [0 1 2 3 4 5 6 7 8]


In [5]:
df = df[df["use"] == True]

df = df.reset_index(drop=True)  # Reset index for future data balancing

print(f"The total number of samples is {len(df)}")

# Ensure invalid data were not existed
extracted_df = df[df["target_x"].isna()]
if extracted_df is not None and not extracted_df.empty:
    raise Exception("Extracted dataframe is not None or not empty!")

extracted_df = extracted_df[:1000]

The total number of samples is 46299


## Dataset Preparation

In [6]:
image_paths = df["image_path"].to_list()

combined_positions = abs(df["motor_a_relative_position"]) + abs(
    df["motor_b_relative_position"]
)
relative_positions = (combined_positions / RELATIVE_POSITION_SCALE).to_list()
courses = df["course"].to_list()
target_xs = df["target_x"].to_list()

X_all = [
    [x, y, z] for x, y, z in zip(image_paths, relative_positions, courses, strict=False)
]
y_all = target_xs

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=6
)

train_set = RegressionDataset(
    inputs=X_train, outputs=y_train, roi=(0, 0, 640, 480), train_course=train_course
)
val_set = RegressionDataset(
    inputs=X_val, outputs=y_val, roi=(0, 0, 640, 480), train_course=train_course
)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=64, shuffle=True)

## Loss Function, Optimizer, Model Initialization

In [9]:
model_label = "nvidia"

criterion = nn.SmoothL1Loss(beta=0.015)  # or nn.MSELoss()
model = NvidiaModelRegression()
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [10]:
# Initialize TensorBoard writer
writer = SummaryWriter()

# Training loop
num_epochs = 1
best_val_loss = float("inf")
best_model_state = None
best_epoch = 0

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_mode_loss = 0.0
    train_control_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        optimizer.zero_grad()
        inputs = [input_tensor.to(device) for input_tensor in inputs]
        labels = labels.to(device)  # Shape: (batch_size, 1) or (batch_size,)
        outputs = model(inputs[0], inputs[1])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Calculate average losses
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [input_tensor.to(device) for input_tensor in inputs]
            labels = labels.to(device)
            outputs = model(inputs[0], inputs[1])
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    # Calculate average validation losses
    avg_val_loss = val_loss / len(val_loader)

    # Check if this is the best model so far
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()  # Deep copy of model state
        best_epoch = epoch + 1
        torch.save(
            model.state_dict(),
            f"../storage/models/{model_label}_reg_{train_course}_{date_label}.pt",
        )
        print(f"  New best model found at epoch {best_epoch}!")

    # Log to TensorBoard
    writer.add_scalar("Loss/train_total", avg_train_loss, epoch)
    writer.add_scalar("Loss/val_total", avg_val_loss, epoch)
    writer.add_scalar("Loss/best_val", best_val_loss, epoch)

    print(f"Epoch {epoch + 1}/{num_epochs}:")
    print(f"  Train - Total: {avg_train_loss:.5f}")
    print(f"  Val   - Total: {avg_val_loss:.5f}")

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(
        f"\nLoaded best model from epoch {best_epoch} with validation loss: {best_val_loss:.5f}"
    )
else:
    print("\nNo best model found, keeping final model state")

# Close TensorBoard writer
writer.close()

print("\nTraining completed! Feature maps have been logged to TensorBoard.")
print("To view the log, run: tensorboard --logdir=runs")

Epoch 1/1: 100%|█████████████████████████████| 290/290 [05:35<00:00,  1.16s/it]


  New best model found at epoch 1!
Epoch 1/1:
  Train - Total: 0.05280
  Val   - Total: 0.01995

Loaded best model from epoch 1 with validation loss: 0.01995

Training completed! Feature maps have been logged to TensorBoard.
To view the log, run: tensorboard --logdir=runs


## Export to ONNX Model

In [11]:
onnx_path = f"../storage/models/{model_label}_reg_{train_course}_{date_label}.onnx"

model.eval()

# Create dummy inputs - adjust dimensions as needed
dummy_image = torch.randn(1, 3, 66, 200)
dummy_relative_position = torch.randn(1, 1)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_image, dummy_relative_position),
    onnx_path,
    export_params=True,
    input_names=["image", "relative_position"],
    output_names=["control_output"],
)

C:\Users\er-ri\AppData\Local\Temp\ipykernel_2192\3491279888.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


## Example Usage

In [12]:
import onnxruntime as ort

# Load the ONNX model
onnx_path = f"../storage/models/{model_label}_reg_{train_course}_{date_label}.onnx"
session = ort.InferenceSession(onnx_path)

# Create dummy inputs (same as during export)
dummy_image = np.random.randn(1, 3, 66, 200).astype(np.float32)
dummy_relative_position = np.random.randn(1, 1).astype(np.float32)

# Prepare inputs dictionary
inputs = {"image": dummy_image, "relative_position": dummy_relative_position}

# Run inference
outputs = session.run(["control_output"], inputs)

# Get results
control_output = outputs[0]

# control_output
print(f"Control output shape: {control_output.shape}")
print(f"Control output: {control_output}")

Control output shape: (1, 1)
Control output: [[0.00183807]]
